## Nature 论文数据抓取与更新
 
_本Notebook实现以下功能：_
-  从现有JSON文件 ( 在Task2中已生成 ) 加载数据
-  更新每篇论文的作者信息
-  将结果保存为最终JSON文件

###  导入必要库

In [11]:
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
from retrying import retry
import random

### 基础配置

In [12]:
# 基础配置参数
BASE_URL = "https://www.nature.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": BASE_URL,
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive"
}
TIMEOUT = 30  # 请求超时时间（秒）
MAX_RETRIES = 3  # 最大重试次数
MIN_DELAY = 1.5  # 最小请求间隔
MAX_DELAY = 4.5  # 最大请求间隔

### 核心功能函数

In [13]:
def retry_on_timeout(exception):
    """定义需要重试的异常类型"""
    return isinstance(exception, (requests.exceptions.Timeout, 
                                 requests.exceptions.ConnectionError))

@retry(stop_max_attempt_number=MAX_RETRIES, 
       retry_on_exception=retry_on_timeout)

def safe_request(url):
    """
    带自动重试机制的请求函数
    参数：
        url (str): 目标URL
    返回：
        requests.Response: 响应对象
    """
    response = requests.get(url, 
                           headers=HEADERS, 
                           timeout=TIMEOUT)
    response.raise_for_status()
    return response

In [14]:
def extract_authors(soup):
    """
    多策略作者提取函数
    参数：
        soup (BeautifulSoup): 解析后的页面对象
    返回：
        list: 作者列表
    """
    authors = []
    
    # 策略1：通过data-test属性精准定位
    author_tags = soup.select('a[data-test="author-name"]')
    if author_tags:
        authors = [tag.get_text(strip=True) for tag in author_tags]
        return _clean_authors(authors)
    
    # 策略2：备用选择器（应对页面结构微调）
    author_tags = soup.select('.c-article-author-list__item a[href^="#auth"]')
    if author_tags:
        authors = [tag.get_text(strip=True) for tag in author_tags]
        return _clean_authors(authors)
    
    # 策略3：结构化数据兜底
    try:
        ld_json = json.loads(soup.find('script', type='application/ld+json').string)
        return [a["name"] for a in ld_json["mainEntity"]["author"]]
    except Exception as e:
        print(f"结构化数据解析失败: {str(e)}")
    
    return authors

def _clean_authors(author_list):
    """数据清洗函数"""
    return [a for a in author_list if a and len(a) > 1]

In [15]:
def update_article_authors(paper):
    """
    单篇文章作者更新流程
    参数：
        paper (dict): 论文数据字典
    返回：
        dict: 更新后的论文数据
    """
    try:
        full_url = urljoin(BASE_URL, paper["url"])
        print(f"\n[{time.strftime('%H:%M:%S')}] 处理论文: {paper['title'][:35]}...")
        
        # 发送请求（带自动重试）
        response = safe_request(full_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 提取最新作者列表
        new_authors = extract_authors(soup)
        original_count = len(paper["authors"])
        
        # 智能更新策略
        if not new_authors:
            print("⚠️ 未获取到作者信息，保留原始数据")
            return paper
            
        if len(new_authors) > original_count:
            paper["authors"] = new_authors
            print(f"✅ 作者更新成功（数量增加 {original_count}→{len(new_authors)}）")
        elif len(new_authors) == original_count:
            if paper["authors"] != new_authors:
                paper["authors"] = new_authors
                print(f"🔄 作者排序/拼写修正（保持 {original_count} 位）")
            else:
                print(f"⏩ 作者信息无变化（保持 {original_count} 位）")
        else:
            print(f"⚠️ 获取到更少作者（{len(new_authors)} 位），建议手动核查")
        
        return paper
        
    except Exception as e:
        print(f"❌ 处理失败: {str(e)}")
        return paper


In [16]:
def save_to_json(data, filename):
    """保存数据到JSON文件"""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"\n数据已保存至 {filename}")


### 主执行流程

In [17]:
# 加载初始数据
try:
    with open('nature_llm_pre.json', 'r', encoding='utf-8') as f:
        journal_data = json.load(f)
    print(f"✅ 成功加载初始数据，包含 {len(journal_data)} 个期刊")
except Exception as e:
    print(f"❌ 数据加载失败: {str(e)}")
    raise


✅ 成功加载初始数据，包含 17 个期刊


In [18]:
# 执行数据更新
start_time = time.time()

for journal in journal_data:
    print(f"\n{'='*40}")
    print(f" 正在处理期刊：{journal['journal']} ")
    print(f" 包含论文数：{len(journal['papers'])} ")
    print(f"{'='*40}")
    
    for i, paper in enumerate(journal["papers"], 1):
        journal["papers"][i-1] = update_article_authors(paper)
        
        # 随机延迟（模拟人工操作）
        delay = random.uniform(MIN_DELAY, MAX_DELAY)
        time.sleep(delay)

total_time = time.time() - start_time
print(f"\n总耗时：{total_time//60:.0f}分{total_time%60:.0f}秒")


 正在处理期刊：Nature Machine Intelligence 
 包含论文数：6 

[22:11:14] 处理论文: LLM-based agentic systems in medici...
✅ 作者更新成功（数量增加 3→8）

[22:11:19] 处理论文: Contextual feature extraction hiera...
✅ 作者更新成功（数量增加 3→5）

[22:11:25] 处理论文: What is in your LLM-based framework...
⚠️ 未获取到作者信息，保留原始数据

[22:11:30] 处理论文: Augmenting large language models wi...
✅ 作者更新成功（数量增加 3→6）

[22:11:36] 处理论文: Multimodal language and graph learn...
✅ 作者更新成功（数量增加 3→5）

[22:11:41] 处理论文: Guidelines for ethical use and ackn...
✅ 作者更新成功（数量增加 3→7）

 正在处理期刊：Nature Communications 
 包含论文数：10 

[22:11:47] 处理论文: LLM-driven multimodal target volume...
✅ 作者更新成功（数量增加 3→7）

[22:11:55] 处理论文: An automatic end-to-end chemical sy...
✅ 作者更新成功（数量增加 3→14）

[22:12:03] 处理论文: Using large language models to acce...
✅ 作者更新成功（数量增加 3→16）

[22:12:10] 处理论文: Matching patients to clinical trial...
✅ 作者更新成功（数量增加 3→10）

[22:12:17] 处理论文: A liquid metal-based module emulati...
⏩ 作者信息无变化（保持 2 位）

[22:12:21] 处理论文: ChatMOF: an artificial intelligence...
⏩ 作者信息无变化（保持 2

### 结果保存与验证

In [19]:
try:
    with open('nature_llm.json', 'w', encoding='utf-8') as f:
        json.dump(journal_data, f, indent=2, ensure_ascii=False)
    print("✅ 数据保存成功：nature_llm.json")
except Exception as e:
    print(f"❌ 保存失败: {str(e)}")

# 验证结果
print("\n更新后首条记录示例：")
sample_paper = journal_data[0]["papers"][0]
print(f"标题：{sample_paper['title']}")
print(f"作者数量：{len(sample_paper['authors'])}")
print("前3位作者：", sample_paper['authors'][:3])

✅ 数据保存成功：nature_llm.json

更新后首条记录示例：
标题：LLM-based agentic systems in medicine and healthcare
作者数量：8
前3位作者： ['Jianing Qiu', 'Kyle Lam', 'Guohao Li']


### 执行说明
1. 确保`nature_llm.json`文件位于当前目录
2. 按顺序执行所有代码单元格
3. 最终结果将保存为`nature_llm_final.json`
4. 完整执行可能需要较长时间（取决于网络速度）